# Reservoirs in Catalunya
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 27-04-2026<br>

**Introduction:**<br>
This code preprocesses the reservoir time series downloaded from [Agència Catalana de l'Aigua](https://analisi.transparenciacatalunya.cat/es/Medi-Ambient/Xarxes-de-control-del-medi-consulta-de-l-aigua-i-e/wc95-u57z/about_data). The raw data includes reservoir attributes (coordinates, ID, name) and daily time series of reservoir storage, level and fraction filled. The results of the code are a CSV file with the reservoir attributes, and several CSV files (one for each reservoir) with the daily timeseries.

> **Note**. Later I've added manually data to the attributes table extracted from the CEDEX dataset.

In [1]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import yaml

# from ocab.config import Config
from ocab.anuario.stations.reservoirs import get_reservoirs_aca

## Configuration

In [2]:
# # load configuration of the BEAVERS dataset
# cfg = Config('../../BEAVERS/config_BEAVERS_v100.yml')

# path where the data is stored
path_aca = Path('/home/casadoj/Data/ACA')

# paths where results will be saved
path_results = path_aca / 'processed' / 'reservoirs'
path_gis = path_results / 'GIS'
path_ts = path_results / 'timeseries'
for path in [path_gis, path_ts]:
    path.mkdir(parents=True, exist_ok=True)

file_map_datasets = Path('map_reservoirs_ACA.yml')

## Extract Raw Data

Here I load and prepocess the raw Excel files downloaded from the ACA website. The result are two objects: `timeseries` is a dictionary that contains the time series for each reservoir, `reservoirs_aca` is a DataFrame with the reservoir attributes.

In [12]:
# time series will be saved in a dictionary and reservoir attributes in a pandas.DataFrame
timeseries = {}
dams = pd.DataFrame()

# read raw Excel files iteratively
path_in = path_aca / 'raw' / 'reservoirs'
files = sorted(list(path_in.glob('reservoirs_*.xlsx')))
for file in tqdm(files, desc='files'):

    # get reservoir attributes and time series
    attrs, ts = get_reservoirs_aca(file)

    # update reservoirs
    dams = pd.concat([dams, attrs], axis=0).drop_duplicates()

    # update time series
    for ID in ts:
        if ID in timeseries:
            timeseries[ID] = pd.concat(
                (timeseries[ID], ts[ID]),
                axis=0
            ).asfreq('D').sort_index(axis=0)
        else:
            timeseries[ID] = ts[ID]

dams.index.name = 'id_saih'

files:   0%|          | 0/8 [00:00<?, ?it/s]

## Attributes

In [13]:
# load mapping between reservoir datasets
if file_map_datasets.is_file():
    # read mapping
    with open(file_map_datasets, 'r') as file:
        map_datasets = yaml.safe_load(file)

    # add codes to the attributes
    keys = list(next(iter(map_datasets.values())).keys())
    cols = {key: f'id_{key.lower()}' for key in keys}
    for key, col in cols.items():
        dams[col] = dams.index.map({
            ID: dct[key] for ID, dct in map_datasets.items()
            })
    dams[list(cols.values())] = dams[list(cols.values())].astype('Int64')

# make up the ID so it's an integer and uses 9000 values as if it was in the Ebro
dams['id'] = [int('90' + x.strip('E')) for x in dams.index]
# dams = dams.reset_index().set_index('id', drop=True)
# dams.sort_index(axis=0, inplace=True)

# add attributes
dams['basin'] = 'CATALUNYA'
# dams['administration'] = 'C.I. CATALUNYA'
for ID in dams.index:
    ts = timeseries[ID]
    start, end = ts.index.min(), ts.index.max()
    dams.loc[ID, ['start', 'end']] = start.year, end.year
    dams.loc[ID, 'active'] = 1 if end.year == 2024 else 0
dams[['start', 'end', 'active']] = dams[['start', 'end', 'active']].astype('Int64')

# sort columns
cols = sorted([col for col in dams.columns if col != 'geometry']) + ['geometry']
dams = dams[cols]

## Export

### Attributes

In [16]:
# export
dams.to_file(path_gis / 'dams_aca.geojson', driver='GeoJSON')

### Time series

In [17]:
for ID, ts in timeseries.items():
    ts.to_parquet(path_ts / f'{ID}.parquet')